In [1]:
import torch
import torch.nn as nn
import torch
import torch.nn as nn
from torch.nn.functional import one_hot
from torch import Tensor
from typing import Union

# FocalLossクラスの定義は省略

class FocalLoss(nn.Module):
    """Computes the focal loss between input and target
    as described here https://arxiv.org/abs/1708.02002v2

    Args:
        gamma (float):  The focal loss focusing parameter.
        weights (Union[None, Tensor]): Rescaling weight given to each class.
        If given, has to be a Tensor of size C. optional.
        reduction (str): Specifies the reduction to apply to the output.
        it should be one of the following 'none', 'mean', or 'sum'.
        default 'mean'.
        ignore_index (int): Specifies a target value that is ignored and
        does not contribute to the input gradient. optional.
        eps (float): smoothing to prevent log from returning inf.
    """
    def __init__(
            self,
            gamma,
            weights: Union[None, Tensor] = None,
            reduction: str = 'mean',
            ignore_index=-100,
            eps=1e-16
            ) -> None:
        super().__init__()
        if reduction not in ['mean', 'none', 'sum']:
            raise NotImplementedError(
                'Reduction {} not implemented.'.format(reduction)
                )
        assert weights is None or isinstance(weights, Tensor), \
            'weights should be of type Tensor or None, but {} given'.format(
                type(weights))
        self.reduction = reduction
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.eps = eps
        self.weights = weights

    def _get_weights(self, target: Tensor) -> Tensor:
        if self.weights is None:
            return torch.ones(target.shape[0])
        weights = target * self.weights
        return weights.sum(dim=-1)

    def _process_target(
            self, target: Tensor, num_classes: int, mask: Tensor
            ) -> Tensor:
        
        #convert all ignore_index elements to zero to avoid error in one_hot
        #note - the choice of value 0 is arbitrary, but it should not matter as these elements will be ignored in the loss calculation
        target = target * (target!=self.ignore_index) 
        target = target.view(-1)
        return one_hot(target, num_classes=num_classes)

    def _process_preds(self, x: Tensor) -> Tensor:
        if x.dim() == 1:
            x = torch.vstack([1 - x, x])
            x = x.permute(1, 0)
            return x
        return x.view(-1, x.shape[-1])

    def _calc_pt(
            self, target: Tensor, x: Tensor, mask: Tensor
            ) -> Tensor:
        p = target * x
        p = p.sum(dim=-1)
        p = p * ~mask
        return p

    def forward(self, x: Tensor, target: Tensor) -> Tensor:
        assert torch.all((x >= 0.0) & (x <= 1.0)), ValueError(
            'The predictions values should be between 0 and 1, \
                make sure to pass the values to sigmoid for binary \
                classification or softmax for multi-class classification'
        )
        mask = target == self.ignore_index
        mask = mask.view(-1)
        x = self._process_preds(x)
        num_classes = x.shape[-1]
        target = self._process_target(target, num_classes, mask)
        weights = self._get_weights(target).to(x.device)
        pt = self._calc_pt(target, x, mask)
        focal = 1 - pt
        nll = -torch.log(self.eps + pt)
        nll = nll.masked_fill(mask, 0)
        loss = weights * (focal ** self.gamma) * nll
        return self._reduce(loss, mask, weights)

    def _reduce(self, x: Tensor, mask: Tensor, weights: Tensor) -> Tensor:
        if self.reduction == 'mean':
            return x.sum() / (~mask * weights).sum()
        elif self.reduction == 'sum':
            return x.sum()
        else:
            return x

# パラメータ設定
gamma = 2.0  # フォーカス強度のパラメータ
weights = torch.tensor([0.5, 2.0, 1.5])  # 各クラスに対する重み（例）

# FocalLoss のインスタンスを生成
criterion = FocalLoss(gamma=gamma, weights=weights, reduction='mean')

# 予測データとターゲットデータの準備
predictions = torch.tensor([[0.1, 0.2, 0.7], [0.3, 0.4, 0.3]], requires_grad=True)
targets = torch.tensor([2, 1])  # クラスインデックスのターゲット

# 損失計算
loss = criterion(predictions, targets)
print("Focal Loss:", loss.item())

# バックプロパゲーション
loss.backward()


Focal Loss: 0.20225155353546143


In [2]:
import torch
from torch import nn

# Focal_MultiLabel_Loss クラスの定義
class Focal_MultiLabel_Loss(nn.Module):
    def __init__(self, gamma):
        super(Focal_MultiLabel_Loss, self).__init__()
        self.gamma = gamma
        self.bceloss = nn.BCELoss(reduction='none')

    def forward(self, outputs, targets): 
        bce = self.bceloss(outputs, targets)
        bce_exp = torch.exp(-bce)
        focal_loss = (1 - bce_exp) ** self.gamma * bce
        return focal_loss.mean()

# 1. Focal_MultiLabel_Loss のインスタンスを生成
gamma = 2.0
criterion = Focal_MultiLabel_Loss(gamma)

# 2. ダミーデータ（予測とターゲット）を作成
# 出力は 2サンプル × 3クラス の確率（0-1）の範囲にします
outputs = torch.tensor([[0.9, 0.2, 0.1], [0.3, 0.7, 0.6]], requires_grad=True)
targets = torch.tensor([[1, 0, 0], [0, 1, 1]], dtype=torch.float32)

# 3. 損失を計算
loss = criterion(outputs, targets)

# 4. 結果を表示
print("Focal Multi-Label Loss:", loss.item())

# 5. バックプロパゲーション
loss.backward()


Focal Multi-Label Loss: 0.026161089539527893


In [3]:
import numpy as np

# Given class weights
class_weights = [
    2.11825747e-05, 5.61270683e-02, 5.61270683e-02, 5.94779380e-02,
    5.69288835e-02, 5.38516466e-02, 5.38516466e-02, 4.98127731e-02,
    5.94779380e-02, 5.61270683e-02, 6.13080284e-02, 5.17535305e-02,
    5.38516466e-02, 6.03791189e-02, 5.69288835e-02, 4.91978006e-02,
    5.45893404e-02, 5.24344980e-02, 5.77539398e-02
]

# Calculating the sum of the class weights
weights_sum = np.sum(class_weights)

print(weights_sum)


0.9999999999746999
